# HuggingFace LLM 텍스트 생성 실습

이 노트북에서는 HuggingFace의 LLM을 불러와 텍스트 생성을 직접 실습합니다.

**학습 목표:**
1. Base 모델과 Instruct 모델의 차이를 이해한다
2. 텍스트 생성 파라미터의 역할을 이해하고 직접 조절해본다

**실행 환경:** Google Colab (T4 GPU 권장)

---
## 1. 환경 설정

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# GPU 확인
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU를 사용할 수 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택해주세요.")

---
## 2. 모델 로드

같은 크기의 **Base 모델**과 **Instruct 모델**을 각각 불러옵니다.

| 구분 | 모델 | 설명 |
|------|------|------|
| Base | `Qwen/Qwen2.5-1.5B` | 대량의 텍스트로 사전학습만 된 모델. 다음 토큰을 예측하도록 학습됨 |
| Instruct | `Qwen/Qwen2.5-1.5B-Instruct` | Base 모델에 지시-응답 데이터로 추가 학습(fine-tuning)한 모델 |

In [ ]:
# Base 모델 로드
base_model_name = "Qwen/Qwen2.5-1.5B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto",
)

print(f"Base 모델 로드 완료: {base_model_name}")

In [ ]:
# Instruct 모델 로드
instruct_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name,
    dtype=torch.float16,
    device_map="auto",
)

print(f"Instruct 모델 로드 완료: {instruct_model_name}")

In [ ]:
def generate_text(model, tokenizer, prompt, **kwargs):
    """모델에 프롬프트를 넣고 텍스트를 생성하는 함수"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    # 기본 생성 설정
    gen_kwargs = {
        "max_new_tokens": 128,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "pad_token_id": tokenizer.eos_token_id,
    }
    gen_kwargs.update(kwargs)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    # 생성된 부분만 디코딩
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated

Instruct 모델은 chat template 형식의 프롬프트를 사용해야 제대로 동작합니다.

In [ ]:
def generate_with_instruct(model, tokenizer, user_message, **kwargs):
    """Instruct 모델용 - chat template을 적용하여 생성"""
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate_text(model, tokenizer, prompt, **kwargs)

---
## 3. Base 모델 vs Instruct 모델 비교

동일한 질문/지시를 두 모델에 넣어보고, 출력이 어떻게 다른지 비교합니다.

- **Base 모델**: 다음에 올 텍스트를 예측 → 입력 텍스트를 자연스럽게 "이어서" 쓰는 경향
- **Instruct 모델**: 지시를 따르도록 학습됨 → 질문에 "답변"하는 경향

### 비교 1: 단순 질문

In [ ]:
question = "인공지능이란 무엇인가?"

print("="*60)
print("[Base 모델]")
print("="*60)
print(generate_text(base_model, base_tokenizer, question))

print("\n" + "="*60)
print("[Instruct 모델]")
print("="*60)
print(generate_with_instruct(instruct_model, instruct_tokenizer, question))

### 비교 2: 번역 지시

In [ ]:
instruction = "다음 문장을 영어로 번역해주세요: 오늘 날씨가 정말 좋습니다."

print("="*60)
print("[Base 모델]")
print("="*60)
print(generate_text(base_model, base_tokenizer, instruction))

print("\n" + "="*60)
print("[Instruct 모델]")
print("="*60)
print(generate_with_instruct(instruct_model, instruct_tokenizer, instruction))

### 비교 3: 요약 지시

In [ ]:
summarize = """다음 글을 한 줄로 요약해주세요:

대규모 언어 모델(LLM)은 방대한 양의 텍스트 데이터를 학습하여 자연어를 이해하고 생성할 수 있는 인공지능 모델입니다. 
이러한 모델은 번역, 요약, 질의응답 등 다양한 자연어 처리 작업에서 뛰어난 성능을 보여주고 있습니다. 
최근에는 ChatGPT, Claude 등 대화형 AI 서비스로 일반 사용자들에게도 널리 알려지게 되었습니다."""

print("="*60)
print("[Base 모델]")
print("="*60)
print(generate_text(base_model, base_tokenizer, summarize))

print("\n" + "="*60)
print("[Instruct 모델]")
print("="*60)
print(generate_with_instruct(instruct_model, instruct_tokenizer, summarize))

### 비교 정리

| 구분 | Base 모델 | Instruct 모델 |
|------|-----------|---------------|
| 학습 방식 | 다음 토큰 예측 (자기회귀) | Base + 지시-응답 데이터로 추가 학습 |
| 입력 처리 | 텍스트를 이어서 작성 | 지시를 이해하고 답변 생성 |
| 출력 특성 | 문맥에 맞는 자연스러운 연속 | 구조화된 답변 |
| 적합한 용도 | 글쓰기, 문장 완성 | 질의응답, 번역, 요약 등 태스크 수행 |

---
## 4. 텍스트 생성 파라미터 설명

LLM이 텍스트를 생성할 때, 다음 토큰을 선택하는 방식을 다양한 파라미터로 조절할 수 있습니다.

### 샘플링 조절 (Sampling Controls)

`do_sample=True`일 때 작동하는 파라미터들입니다.

| 파라미터 | 설명 | 값 범위 | 효과 |
|----------|------|---------|------|
| `temperature` | 확률 분포의 날카로움 조절 | 0 초과 ~ 무한대 | **낮을수록**(→0) 확률이 높은 토큰에 집중 (확정적, 반복적). **높을수록**(→2+) 확률 분포가 균일해짐 (다양하지만 횡설수설 가능) |
| `top_k` | 상위 k개 토큰만 후보로 사용 | 1 이상의 정수 | 확률 상위 k개 토큰 중에서만 샘플링. 작을수록 안전하고, 클수록 다양함 |
| `top_p` | 누적 확률 p까지의 토큰만 후보로 사용 | 0.0 ~ 1.0 | 확률을 높은 순으로 누적하여 p에 도달할 때까지의 토큰만 후보로 사용 (Nucleus Sampling). 0.1이면 매우 보수적, 0.9면 다양한 표현 |

### 생성 길이 및 반복 방지

| 파라미터 | 설명 | 값 범위 | 효과 |
|----------|------|---------|------|
| `max_new_tokens` | 생성할 최대 토큰 수 | 1 이상의 정수 | 생성 길이 상한. 모델이 EOS 토큰을 먼저 생성하면 그 전에 멈춤 |
| `repetition_penalty` | 반복 생성 패널티 | 1.0 이상 | 1.0이면 패널티 없음. 1.2~1.5 정도면 이미 생성된 토큰이 다시 선택될 확률이 줄어듦. 너무 높으면 문맥이 부자연스러워짐 |

### 파라미터 간의 관계

```
모델이 다음 토큰의 확률 분포를 계산
        │
        ▼
  temperature 적용 (분포의 날카로움 조절)
        │
        ▼
  top_k 필터링 (상위 k개만 남김)
        │
        ▼
  top_p 필터링 (누적 확률 p까지만 남김)
        │
        ▼
  repetition_penalty 적용 (이미 나온 토큰 확률 감소)
        │
        ▼
  최종 토큰 선택 (sampling 또는 greedy)
```

---
## 5. 파라미터 직접 실험

이제 파라미터를 직접 바꿔가며 생성 결과가 어떻게 달라지는지 실험해봅시다.

Instruct 모델을 사용합니다.

### 실험 1: temperature 비교

temperature를 낮추면 더 확정적이고 반복적인 답변, 높이면 더 다양하지만 횡설수설할 수 있습니다.

In [ ]:
prompt = "봄이 오면"

for temp in [0.1, 0.5, 1.0, 1.5]:
    print(f"\n{'='*60}")
    print(f"temperature = {temp}")
    print("="*60)
    result = generate_with_instruct(
        instruct_model, instruct_tokenizer, prompt,
        temperature=temp, top_p=0.95, max_new_tokens=100
    )
    print(result)

아래 코드로 상위 10개 단어들이 선택될 확률을 시각화 해 볼 수 있습니다.

In [ ]:
def show_top_tokens(prompt, temperature=1.0):
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    
    with torch.no_grad():
        outputs = base_model(**inputs)
    
    # 마지막 토큰 위치의 logit
    logits = outputs.logits[0, -1, :]
    logits_temp = logits / temperature          # temperature 적용
    probs = torch.softmax(logits_temp, dim=-1)
    
    top5 = torch.topk(probs, 10)
    print(f"\n[temperature={temperature}]")
    for score, idx in zip(top5.values, top5.indices):
        token = base_tokenizer.decode([idx])
        bar = "█" * int(score.item() * 50)
        print(f"  {token!r:15} {bar} {score.item():.3f}")

show_top_tokens("오늘 날씨가", temperature=0.1)
show_top_tokens("오늘 날씨가", temperature=1.0)
show_top_tokens("오늘 날씨가", temperature=2.0)

### 실험 2: top_p 비교

top_p가 작을수록 확률이 높은 소수의 토큰만 사용하고, 클수록 다양한 토큰을 후보로 사용합니다.

In [ ]:
prompt = "인공지능의 미래에 대해 설명해주세요."

for p in [0.1, 0.5, 0.9]:
    print(f"\n{'='*60}")
    print(f"top_p = {p}")
    print("="*60)
    result = generate_with_instruct(
        instruct_model, instruct_tokenizer, prompt,
        temperature=0.7, top_p=p, max_new_tokens=100
    )
    print(result)

### 실험 3: top_k 비교

top_k가 작으면 가장 유력한 몇 개의 토큰만 사용하고, 크면 더 넓은 범위에서 선택합니다.

In [ ]:
prompt = "맛있는 파스타를 만드는 방법을 알려주세요."

for k in [5, 20, 50]:
    print(f"\n{'='*60}")
    print(f"top_k = {k}")
    print("="*60)
    result = generate_with_instruct(
        instruct_model, instruct_tokenizer, prompt,
        temperature=0.7, top_k=k, top_p=1.0, max_new_tokens=100
    )
    print(result)

### 실험 4: repetition_penalty 비교

반복 패널티가 없으면 같은 표현이 반복될 수 있고, 너무 높으면 부자연스러워집니다.

In [ ]:
prompt = "한국의 사계절에 대해 설명해주세요."

for penalty in [1.0, 1.2, 1.5]:
    print(f"\n{'='*60}")
    print(f"repetition_penalty = {penalty}")
    print("="*60)
    result = generate_with_instruct(
        instruct_model, instruct_tokenizer, prompt,
        temperature=0.7, top_p=0.9, repetition_penalty=penalty, max_new_tokens=150
    )
    print(result)

### 실험 6: max_new_tokens 비교

생성 길이를 조절하면 답변의 상세함이 달라집니다.

In [ ]:
prompt = "머신러닝과 딥러닝의 차이점을 설명해주세요."

for tokens in [30, 100, 256]:
    print(f"\n{'='*60}")
    print(f"max_new_tokens = {tokens}")
    print("="*60)
    result = generate_with_instruct(
        instruct_model, instruct_tokenizer, prompt,
        temperature=0.7, top_p=0.9, max_new_tokens=tokens
    )
    print(result)

---
## 6. 자유 실험

아래 셀에서 프롬프트와 파라미터를 자유롭게 바꿔가며 실험해보세요!

**용도별 추천 세팅 참고:**

| 용도 | temperature | top_p | top_k | repetition_penalty | 비고 |
|------|-------------|-------|-------|--------------------|----- |
| 정확한 답변 (번역, 요약) | 0.1~0.3 | 0.5~0.7 | 10~20 | 1.0 | 낮은 temperature로 확정적 답변 |
| 일반 대화 | 0.5~0.8 | 0.8~0.95 | 40~50 | 1.1 | 적당한 다양성 |
| 창의적 글쓰기 (시, 소설) | 0.9~1.2 | 0.9~0.95 | 50~100 | 1.2 | 높은 다양성, 반복 방지 |

In [ ]:
# 자유 실험 - 프롬프트와 파라미터를 자유롭게 수정해보세요!

my_prompt = "여기에 원하는 프롬프트를 입력하세요"

result = generate_with_instruct(
    instruct_model, instruct_tokenizer, my_prompt,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    max_new_tokens=200,
)

print(result)

In [ ]:
# Base 모델로도 실험해보세요! (chat template 없이 직접 텍스트를 이어쓰기)

my_prompt = "옛날 옛적에 한 마을에"

result = generate_text(
    base_model, base_tokenizer, my_prompt,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.2,
    max_new_tokens=200,
)

print(result)